# Trabajo 3: Análisis de datos con Numpy y Pandas

**Empresa:** RetailNow

Análisis de ventas, inventarios y satisfacción del cliente de la red de tiendas, para apoyar decisiones sobre el rendimiento de cada sucursal.

Archivos utilizados:
- `sales.csv`: ventas por tienda y producto.
- `inventories.csv`: stock disponible por tienda y producto.
- `satisfaction.csv`: satisfacción promedio de los clientes por tienda.

## 1. Importar librerías

Se usa Pandas para leer y manipular los datos y Numpy para los cálculos estadísticos y la simulación.

In [1]:
import os
import pandas as pd
import numpy as np

print("Librerías importadas correctamente.")
print("Pandas:", pd.__version__, "| Numpy:", np.__version__)

Librerías importadas correctamente.
Pandas: 3.1.0 | Numpy: 2.5.4


## 2. Carga y limpieza de datos

Se cargan los tres CSV en DataFrames distintos usando las rutas absolutas pedidas (`/workspace/...`).
Si se ejecuta fuera de ese entorno, se usan los archivos de la carpeta del proyecto.

Después se eliminan las filas con valores nulos con `dropna()`.

In [5]:
# Ruta pedida en el enunciado; si no existe, se usa la carpeta actual del notebook
RUTA_DATOS = "/workspace"
if not os.path.exists(os.path.join(RUTA_DATOS, "sales.csv")):
    RUTA_DATOS = os.getcwd()

sales_df = pd.read_csv(os.path.join(RUTA_DATOS, "sales.csv"))
inventories_df = pd.read_csv(os.path.join(RUTA_DATOS, "inventories.csv"))
satisfaction_df = pd.read_csv(os.path.join(RUTA_DATOS, "satisfaction.csv"))

print("Datos cargados desde:", RUTA_DATOS)

Datos cargados desde: c:\REPO_UNIR\NumpyyPanda


In [6]:
# Revisar valores nulos antes de limpiar
print("Nulos en ventas:", sales_df.isnull().sum().sum())
print("Nulos en inventarios:", inventories_df.isnull().sum().sum())
print("Nulos en satisfacción:", satisfaction_df.isnull().sum().sum())

# Eliminar filas con valores nulos
sales_df = sales_df.dropna()
inventories_df = inventories_df.dropna()
satisfaction_df = satisfaction_df.dropna()

print("\nFilas después de limpiar:")
print(f"Ventas: {sales_df.shape[0]} | Inventarios: {inventories_df.shape[0]} | Satisfacción: {satisfaction_df.shape[0]}")

Nulos en ventas: 0
Nulos en inventarios: 0
Nulos en satisfacción: 0

Filas después de limpiar:
Ventas: 10 | Inventarios: 10 | Satisfacción: 5


### Verificación de la estructura

Se revisan los tipos de datos de cada columna y se convierten las fechas a tipo `datetime`
para que los DataFrames queden listos para el análisis.

In [7]:
# Convertir columnas de fecha a datetime
sales_df["Fecha_Venta"] = pd.to_datetime(sales_df["Fecha_Venta"])
inventories_df["Fecha_Actualización"] = pd.to_datetime(inventories_df["Fecha_Actualización"])
satisfaction_df["Fecha_Evaluación"] = pd.to_datetime(satisfaction_df["Fecha_Evaluación"])

print("--- Ventas ---")
print(sales_df.dtypes, "\n")
print("--- Inventarios ---")
print(inventories_df.dtypes, "\n")
print("--- Satisfacción ---")
print(satisfaction_df.dtypes)

--- Ventas ---
ID_Tienda                    int64
Producto                       str
Cantidad_Vendida             int64
Precio_Unitario              int64
Fecha_Venta         datetime64[us]
dtype: object 

--- Inventarios ---
ID_Tienda                       int64
Producto                          str
Stock_Disponible                int64
Fecha_Actualización    datetime64[us]
dtype: object 

--- Satisfacción ---
ID_Tienda                         int64
Satisfacción_Promedio             int64
Fecha_Evaluación         datetime64[us]
dtype: object


In [8]:
sales_df.head()

,ID_Tienda,Producto,Cantidad_Vendida,Precio_Unitario,Fecha_Venta
0,1,Producto A,20,100,2023-01-05
1,1,Producto B,15,200,2023-01-06
2,2,Producto A,30,100,2023-01-07
3,2,Producto C,25,300,2023-01-08
4,3,Producto A,10,100,2023-01-09


In [9]:
inventories_df.head()

,ID_Tienda,Producto,Stock_Disponible,Fecha_Actualización
0,1,Producto A,50,2023-01-05
1,1,Producto B,40,2023-01-06
2,2,Producto A,60,2023-01-07
3,2,Producto C,45,2023-01-08
4,3,Producto A,30,2023-01-09


In [10]:
satisfaction_df.head()

,ID_Tienda,Satisfacción_Promedio,Fecha_Evaluación
0,1,85,2023-01-15
1,2,90,2023-01-15
2,3,70,2023-01-15
3,4,65,2023-01-15
4,5,55,2023-01-15


## 3. Exploración de ventas

Se calcula el ingreso de cada venta (`Total_Ventas` = cantidad vendida × precio unitario)
y luego se agrupa con `groupby()` por producto y por tienda.

In [11]:
# Ingreso de cada venta
sales_df["Total_Ventas"] = sales_df["Cantidad_Vendida"] * sales_df["Precio_Unitario"]
sales_df

,ID_Tienda,Producto,Cantidad_Vendida,Precio_Unitario,Fecha_Venta,Total_Ventas
0,1,Producto A,20,100,2023-01-05,2000
1,1,Producto B,15,200,2023-01-06,3000
2,2,Producto A,30,100,2023-01-07,3000
3,2,Producto C,25,300,2023-01-08,7500
4,3,Producto A,10,100,2023-01-09,1000
5,3,Producto B,40,200,2023-01-10,8000
6,4,Producto C,35,300,2023-01-11,10500
7,4,Producto A,25,100,2023-01-12,2500
8,5,Producto B,20,200,2023-01-13,4000
9,5,Producto C,30,300,2023-01-14,9000


In [12]:
# Ventas totales por producto (unidades e ingresos)
ventas_por_producto = sales_df.groupby("Producto")[["Cantidad_Vendida", "Total_Ventas"]].sum()
print("--- Ventas totales por producto ---")
print(ventas_por_producto)

--- Ventas totales por producto ---
            Cantidad_Vendida  Total_Ventas
Producto                                  
Producto A                85          8500
Producto B                75         15000
Producto C                90         27000


In [13]:
# Ventas totales e ingresos por tienda
ventas_por_tienda = sales_df.groupby("ID_Tienda")[["Cantidad_Vendida", "Total_Ventas"]].sum().reset_index()
print("--- Ventas e ingresos totales por tienda ---")
print(ventas_por_tienda)

--- Ventas e ingresos totales por tienda ---
   ID_Tienda  Cantidad_Vendida  Total_Ventas
0          1                35          5000
1          2                55         10500
2          3                50          9000
3          4                60         13000
4          5                50         13000


In [14]:
# Ventas por tienda y producto
ventas_tienda_producto = sales_df.groupby(["ID_Tienda", "Producto"])[["Cantidad_Vendida", "Total_Ventas"]].sum()
print("--- Ventas por tienda y producto ---")
print(ventas_tienda_producto)

--- Ventas por tienda y producto ---
                      Cantidad_Vendida  Total_Ventas
ID_Tienda Producto                                  
1         Producto A                20          2000
          Producto B                15          3000
2         Producto A                30          3000
          Producto C                25          7500
3         Producto A                10          1000
          Producto B                40          8000
4         Producto A                25          2500
          Producto C                35         10500
5         Producto B                20          4000
          Producto C                30          9000


### Resumen estadístico

`describe()` da la cantidad de registros, media, desviación, mínimo, cuartiles (el 50% es la mediana) y máximo.

In [15]:
print("--- Resumen estadístico de las ventas ---")
print(sales_df[["Cantidad_Vendida", "Precio_Unitario", "Total_Ventas"]].describe())

--- Resumen estadístico de las ventas ---
       Cantidad_Vendida  Precio_Unitario  Total_Ventas
count         10.000000        10.000000     10.000000
mean          25.000000       190.000000   5050.000000
std            9.128709        87.559504   3361.960407
min           10.000000       100.000000   1000.000000
25%           20.000000       100.000000   2625.000000
50%           25.000000       200.000000   3500.000000
75%           30.000000       275.000000   7875.000000
max           40.000000       300.000000  10500.000000


### Promedio de ventas por tienda y categoría

Los datos no tienen una columna de categoría, así que se toma el **producto** (A, B, C) como categoría.

In [16]:
promedio_tienda_categoria = sales_df.groupby(["ID_Tienda", "Producto"])["Cantidad_Vendida"].mean().unstack()
print("--- Promedio de unidades vendidas por tienda y producto ---")
print(promedio_tienda_categoria)

print("\n--- Promedio de unidades vendidas por producto ---")
print(sales_df.groupby("Producto")["Cantidad_Vendida"].mean())

--- Promedio de unidades vendidas por tienda y producto ---
Producto   Producto A  Producto B  Producto C
ID_Tienda                                    
1                20.0        15.0         NaN
2                30.0         NaN        25.0
3                10.0        40.0         NaN
4                25.0         NaN        35.0
5                 NaN        20.0        30.0

--- Promedio de unidades vendidas por producto ---
Producto
Producto A    21.25
Producto B    25.00
Producto C    30.00
Name: Cantidad_Vendida, dtype: float64


## 4. Análisis de inventarios

**Rotación de inventario** = unidades vendidas / stock disponible.

El archivo de inventarios no tiene las ventas, por eso se une con el de ventas usando la tienda y el producto (`merge`).
La rotación de cada producto se guarda en una nueva columna del DataFrame de inventarios.

In [17]:
# Unir las unidades vendidas al DataFrame de inventarios
ventas_unidades = sales_df.groupby(["ID_Tienda", "Producto"])["Cantidad_Vendida"].sum().reset_index()
inventories_df = inventories_df.merge(ventas_unidades, on=["ID_Tienda", "Producto"], how="left")

# Si un producto no tuvo ventas, se deja en 0
inventories_df["Cantidad_Vendida"] = inventories_df["Cantidad_Vendida"].fillna(0)

# Rotación por producto (nueva columna)
inventories_df["Rotacion_Inventario"] = inventories_df["Cantidad_Vendida"] / inventories_df["Stock_Disponible"]
inventories_df

,ID_Tienda,Producto,Stock_Disponible,Fecha_Actualización,Cantidad_Vendida,Rotacion_Inventario
0,1,Producto A,50,2023-01-05,20,0.400000
1,1,Producto B,40,2023-01-06,15,0.375000
2,2,Producto A,60,2023-01-07,30,0.500000
3,2,Producto C,45,2023-01-08,25,0.555556
4,3,Producto A,30,2023-01-09,10,0.333333
5,3,Producto B,80,2023-01-10,40,0.500000
6,4,Producto C,70,2023-01-11,35,0.500000
7,4,Producto A,50,2023-01-12,25,0.500000
8,5,Producto B,40,2023-01-13,20,0.500000
9,5,Producto C,60,2023-01-14,30,0.500000


In [18]:
# Rotación por tienda: total vendido / total en stock
rotacion_por_tienda = inventories_df.groupby("ID_Tienda")[["Cantidad_Vendida", "Stock_Disponible"]].sum()
rotacion_por_tienda["Rotacion_Inventario"] = rotacion_por_tienda["Cantidad_Vendida"] / rotacion_por_tienda["Stock_Disponible"]
rotacion_por_tienda["Porcentaje_Vendido"] = (rotacion_por_tienda["Rotacion_Inventario"] * 100).round(2)

print("--- Rotación de inventario por tienda ---")
print(rotacion_por_tienda)

--- Rotación de inventario por tienda ---
           Cantidad_Vendida  Stock_Disponible  Rotacion_Inventario  \
ID_Tienda                                                            
1                        35                90             0.388889   
2                        55               105             0.523810   
3                        50               110             0.454545   
4                        60               120             0.500000   
5                        50               100             0.500000   

           Porcentaje_Vendido  
ID_Tienda                      
1                       38.89  
2                       52.38  
3                       45.45  
4                       50.00  
5                       50.00  


### Tiendas con inventario crítico

Se consideran críticas las tiendas (o productos) donde lo vendido es **menor al 10%** del stock disponible (rotación < 0.10).

In [19]:
UMBRAL_CRITICO = 0.10

# Filtro por producto dentro de cada tienda
productos_criticos = inventories_df[inventories_df["Rotacion_Inventario"] < UMBRAL_CRITICO]

# Filtro por tienda
tiendas_criticas = rotacion_por_tienda[rotacion_por_tienda["Rotacion_Inventario"] < UMBRAL_CRITICO]

print("--- Productos con inventario crítico (< 10%) ---")
if productos_criticos.empty:
    print("Ningún producto tiene ventas menores al 10% de su stock.")
else:
    print(productos_criticos[["ID_Tienda", "Producto", "Stock_Disponible", "Cantidad_Vendida", "Rotacion_Inventario"]])

print("\n--- Tiendas con inventario crítico (< 10%) ---")
if tiendas_criticas.empty:
    print("Ninguna tienda tiene un nivel crítico de inventario.")
else:
    print(tiendas_criticas)

# Tienda con menor rotación, como referencia
tienda_menor = rotacion_por_tienda["Rotacion_Inventario"].idxmin()
print(f"\nLa tienda con menor rotación es la {tienda_menor} "
      f"({rotacion_por_tienda.loc[tienda_menor, 'Porcentaje_Vendido']}% vendido).")

--- Productos con inventario crítico (< 10%) ---
Ningún producto tiene ventas menores al 10% de su stock.

--- Tiendas con inventario crítico (< 10%) ---
Ninguna tienda tiene un nivel crítico de inventario.

La tienda con menor rotación es la 1 (38.89% vendido).


## 5. Satisfacción del cliente

Se une la satisfacción con las ventas e ingresos de cada tienda para ver si hay relación entre ambas.

In [20]:
# Unir satisfacción con ventas por tienda
satisfaccion_ventas = satisfaction_df.merge(ventas_por_tienda, on="ID_Tienda")
satisfaccion_ventas = satisfaccion_ventas.merge(
    rotacion_por_tienda[["Rotacion_Inventario"]].reset_index(), on="ID_Tienda"
)
satisfaccion_ventas = satisfaccion_ventas.sort_values("Satisfacción_Promedio", ascending=False)

print("--- Satisfacción vs. ventas por tienda ---")
print(satisfaccion_ventas[["ID_Tienda", "Satisfacción_Promedio", "Cantidad_Vendida", "Total_Ventas", "Rotacion_Inventario"]])

# Correlación entre satisfacción y ventas
corr_unidades = satisfaccion_ventas["Satisfacción_Promedio"].corr(satisfaccion_ventas["Cantidad_Vendida"])
corr_ingresos = satisfaccion_ventas["Satisfacción_Promedio"].corr(satisfaccion_ventas["Total_Ventas"])
print(f"\nCorrelación satisfacción - unidades vendidas: {corr_unidades:.2f}")
print(f"Correlación satisfacción - ingresos: {corr_ingresos:.2f}")

--- Satisfacción vs. ventas por tienda ---
   ID_Tienda  Satisfacción_Promedio  Cantidad_Vendida  Total_Ventas  \
1          2                     90                55         10500   
0          1                     85                35          5000   
2          3                     70                50          9000   
3          4                     65                60         13000   
4          5                     55                50         13000   

   Rotacion_Inventario  
1             0.523810  
0             0.388889  
2             0.454545  
3             0.500000  
4             0.500000  

Correlación satisfacción - unidades vendidas: -0.32
Correlación satisfacción - ingresos: -0.66


In [21]:
# Tiendas con satisfacción baja (< 60%)
UMBRAL_SATISFACCION = 60

tiendas_baja_satisfaccion = satisfaccion_ventas[satisfaccion_ventas["Satisfacción_Promedio"] < UMBRAL_SATISFACCION]

print("--- Tiendas con satisfacción menor al 60% ---")
if tiendas_baja_satisfaccion.empty:
    print("Ninguna tienda tiene satisfacción menor al 60%.")
else:
    print(tiendas_baja_satisfaccion[["ID_Tienda", "Satisfacción_Promedio", "Cantidad_Vendida", "Total_Ventas"]])

--- Tiendas con satisfacción menor al 60% ---
   ID_Tienda  Satisfacción_Promedio  Cantidad_Vendida  Total_Ventas
4          5                     55                50         13000


In [22]:
# Recomendaciones para las tiendas con satisfacción baja
promedio_ingresos = ventas_por_tienda["Total_Ventas"].mean()

for _, tienda in tiendas_baja_satisfaccion.iterrows():
    print(f"Tienda {tienda['ID_Tienda']} - satisfacción {tienda['Satisfacción_Promedio']}%")
    print(f"  Ingresos: ${tienda['Total_Ventas']:,.0f} (promedio de la red: ${promedio_ingresos:,.0f})")
    print("  Recomendaciones:")
    print("  - Aplicar encuestas cortas para conocer las causas de la insatisfacción.")
    print("  - Capacitar al personal en atención al cliente.")
    print("  - Revisar tiempos de espera y disponibilidad de productos.")
    print("  - Tomar como referencia las prácticas de las tiendas mejor evaluadas.")

Tienda 5 - satisfacción 55%
  Ingresos: $13,000 (promedio de la red: $10,100)
  Recomendaciones:
  - Aplicar encuestas cortas para conocer las causas de la insatisfacción.
  - Capacitar al personal en atención al cliente.
  - Revisar tiempos de espera y disponibilidad de productos.
  - Tomar como referencia las prácticas de las tiendas mejor evaluadas.


### Conclusiones de la satisfacción

- La tienda **5** es la única por debajo del 60% (55%).
- La correlación entre satisfacción e ingresos sale negativa (-0.66): la tienda 5 tiene la peor satisfacción pero empata con la 4 en los mayores ingresos. Con solo 5 tiendas no se puede afirmar que exista una relación real.
- Que la tienda 5 venda bien hoy no asegura que siga así; si los clientes no están conformes pueden dejar de comprar, por eso conviene aplicar las recomendaciones.

## 6. Operaciones con Numpy

Se convierte la columna `Total_Ventas` (ingresos totales por tienda) a un array de Numpy con `.to_numpy()`
y se calculan la mediana y la desviación estándar con funciones de Numpy.

In [23]:
ventas_array = ventas_por_tienda["Total_Ventas"].to_numpy()
print("Array de ventas totales por tienda:", ventas_array)

mediana_ventas = np.median(ventas_array)
desviacion_ventas = np.std(ventas_array)
media_ventas = np.mean(ventas_array)

print(f"\nMedia de las ventas totales: ${media_ventas:,.2f}")
print(f"Mediana de las ventas totales: ${mediana_ventas:,.2f}")
print(f"Desviación estándar de las ventas totales: ${desviacion_ventas:,.2f}")

Array de ventas totales por tienda: [ 5000 10500  9000 13000 13000]

Media de las ventas totales: $10,100.00
Mediana de las ventas totales: $10,500.00
Desviación estándar de las ventas totales: $2,973.21


### Simulación de ventas futuras

Se simulan las ventas de los próximos 6 meses para cada tienda con una distribución normal:
la media es la venta actual de cada tienda y la variación es del 15%.
Se fija una semilla (`seed`) para que los resultados se puedan repetir.

In [24]:
np.random.seed(42)

MESES = 6
VARIACION = 0.15

# Matriz de 6 filas (meses) x 5 columnas (tiendas)
proyecciones = np.random.normal(loc=ventas_array, scale=ventas_array * VARIACION, size=(MESES, len(ventas_array)))

# Las ventas no pueden ser negativas
proyecciones = np.clip(proyecciones, 0, None).round(0)

proyecciones_df = pd.DataFrame(
    proyecciones,
    index=[f"Mes {i}" for i in range(1, MESES + 1)],
    columns=[f"Tienda {t}" for t in ventas_por_tienda["ID_Tienda"]],
)
proyecciones_df

,Tienda 1,Tienda 2,Tienda 3,Tienda 4,Tienda 5
Mes 1,5373.0,10282.0,9874.0,15970.0,12543.0
Mes 2,4824.0,12987.0,10036.0,12085.0,14058.0
Mes 3,4652.0,9766.0,9327.0,9269.0,9636.0
Mes 4,4578.0,8905.0,9424.0,11229.0,10246.0
Mes 5,6099.0,10144.0,9091.0,10222.0,11938.0
Mes 6,5083.0,8687.0,9507.0,11829.0,12431.0


In [25]:
# Estadísticas básicas de las proyecciones por tienda
resumen_proyecciones = pd.DataFrame({
    "Venta_Actual": ventas_array,
    "Media_Proyectada": np.mean(proyecciones, axis=0).round(2),
    "Mediana_Proyectada": np.median(proyecciones, axis=0).round(2),
    "Desviacion": np.std(proyecciones, axis=0).round(2),
    "Minimo": np.min(proyecciones, axis=0),
    "Maximo": np.max(proyecciones, axis=0),
}, index=proyecciones_df.columns)
print("--- Estadísticas de las proyecciones por tienda ---")
print(resumen_proyecciones)

# Estadísticas generales de toda la simulación
print(f"\nMedia general proyectada: ${np.mean(proyecciones):,.2f}")
print(f"Desviación estándar general: ${np.std(proyecciones):,.2f}")
print(f"Total proyectado de la red en {MESES} meses: ${np.sum(proyecciones):,.2f}")

--- Estadísticas de las proyecciones por tienda ---
          Venta_Actual  Media_Proyectada  Mediana_Proyectada  Desviacion  \
Tienda 1          5000           5101.50              4953.5      519.96   
Tienda 2         10500          10128.50              9955.0     1408.37   
Tienda 3          9000           9543.17              9465.5      321.20   
Tienda 4         13000          11767.33             11529.0     2108.97   
Tienda 5         13000          11808.67             12184.5     1481.28   

          Minimo   Maximo  
Tienda 1  4578.0   6099.0  
Tienda 2  8687.0  12987.0  
Tienda 3  9091.0  10036.0  
Tienda 4  9269.0  15970.0  
Tienda 5  9636.0  14058.0  

Media general proyectada: $9,669.83
Desviación estándar general: $2,795.33
Total proyectado de la red en 6 meses: $290,095.00


## 7. Conclusiones generales

- **Ventas:** las tiendas 4 y 5 son las que más ingresos generan ($13.000 cada una), seguidas de la 2. Las tres venden el Producto C, que es el de mayor precio y el que más ingresos deja en total ($27.000). La tienda 1 es la de menores ingresos ($5.000).
- **Inventario:** ninguna tienda ni producto está por debajo del 10% de rotación, así que no hay niveles críticos. La tienda con menor rotación es la 1 (38,89%) y el producto con menor rotación es el Producto A de la tienda 3 (33%); conviene vigilarlos.
- **Satisfacción:** la tienda 5 está por debajo del 60% y necesita acciones de mejora en atención al cliente.
- **Proyección:** la simulación con Numpy da un rango de ventas esperadas por tienda que puede servir para planificar inventario y metas.